# Install

In [6]:
#conda activate extrator

In [9]:
pip install pdfplumber pandas tqdm

  Using cached tqdm-4.70.0-py3-none-any.whl.metadata (57 kB)
Using cached tqdm-4.70.0-py3-none-any.whl (80 kB)
Note: you may need to restart the kernel to use updated packages.


## Extrai todas 

In [11]:
import pdfplumber
import pandas as pd
import re
import os
import glob
from tqdm import tqdm

CAMINHO_CSV = "resultados_matriculados_geral.csv"

def extrair_todos_pdfs(pasta="."):
    pdf_files = glob.glob(os.path.join(pasta, "*.pdf"))
    
    if not pdf_files:
        print("Nenhum arquivo PDF encontrado. Certifique-se de que eles estão na mesma pasta que o script.")
        return

    resultados = []

    for pdf_path in pdf_files:
        nome_arquivo = os.path.basename(pdf_path)
        print(f"\nProcessando: {nome_arquivo}...")
        texto_completo = ""
        try:
            with pdfplumber.open(pdf_path) as pdf:
                # Mudança: leave=True mantém a barra de páginas na tela quando ela atinge 100%
                for page in tqdm(pdf.pages, desc="Lendo páginas", unit="pág", leave=True):
                    T = page.extract_text()
                    if T: 
                        texto_completo += T + "\n"
        except Exception as E:
            print(f"Erro ao ler {pdf_path}: {E}")
            continue

        # Novo Padrão de Separação: o OCR dos PDFs pode suprimir o pipe (|) ou a palavra Campus.
        # Agora ele quebra exatamente em 'Curso ofertante:' de forma segura.
        chunks = re.split(r'(?:Campus:\s*.*?\s*(?:\|)?\s*)?Curso ofertante:', texto_completo, flags=re.IGNORECASE)
        
        total_materias = len(chunks) - 1
        if total_materias <= 0:
            print("Nenhuma matéria encontrada neste PDF.")
            continue

        pbar = tqdm(range(total_materias), desc="Analisando matérias", unit="mat")
        
        for I in pbar:
            linhas_chunk_atual = [linha.strip() for linha in chunks[I].split('\n') if linha.strip()]
            if not linhas_chunk_atual: continue
            
            cod_turma = ""
            nome_disc = ""
            
            # Percorre o bloco de baixo para cima para encontrar a disciplina exata 
            # (evita capturar o cabeçalho 'Resultados da classificação' por acidente)
            for linha in reversed(linhas_chunk_atual):
                match = re.search(r'\b([A-Z0-9]{6,})\s*-\s*(\d{2}[A-Z]{2})\b\s*[-]?\s*(.*)', linha)
                if match:
                    cod_turma = f"{match.group(1)}-{match.group(2)}"
                    nome_disc = match.group(3).strip()
                    break

            if not cod_turma:
                linha_disciplina = linhas_chunk_atual[-1]
                partes = linha_disciplina.split('-', 2)
                if len(partes) >= 3:
                    cod_turma = partes[0].strip() + "-" + partes[1].strip()
                    nome_disc = "-".join(partes[2:]).strip()
                else:
                    partes = linha_disciplina.split(' ', 1)
                    cod_turma = partes[0].strip()
                    nome_disc = partes[1].strip() if len(partes) > 1 else ""

            texto_alunos = chunks[I+1]
            ra_matches = list(re.finditer(r'\b(11\d{9}|21\d{6})\b', texto_alunos))

            for J in range(len(ra_matches)):
                ra = ra_matches[J].group(1)
                start_idx = ra_matches[J].end()
                end_idx = ra_matches[J+1].start() if J < len(ra_matches) - 1 else start_idx + 400
                
                # Transforma o bloco do aluno em minúsculo para contornar falhas de OCR
                bloco_aluno = texto_alunos[start_idx:end_idx].lower()

                # O PDF tem falhas graves de extração (ex: "Matriculado" vira "Mat" ou "Mist")
                is_indeferido = re.search(r'\b(indeferido|indaferido|ind|inc)\b', bloco_aluno)
                is_transferido = 'transferido' in bloco_aluno
                is_matriculado = re.search(r'\b(matriculado|matrículado|mat|mist)\b', bloco_aluno)

                if is_matriculado and not is_indeferido and not is_transferido:
                    resultados.append({"RA": ra, "Código_turma": cod_turma, "Nome_Disciplina": nome_disc})

            pbar.set_postfix(cadastros=len(resultados))

    df = pd.DataFrame(resultados)
    if not df.empty:
        df = df[df['RA'].str.strip() != ""]
        df = df.drop_duplicates()
        df.to_csv(CAMINHO_CSV, index=False, sep=';', encoding='utf-8-sig')
        print(f"\n✅ SUCESSO! [{len(df)}] Registros totais extraídos para o arquivo '{CAMINHO_CSV}'.")
    else:
        print("\nNenhum registro validado encontrado nos PDFs.")

if __name__ == '__main__':
    extrair_todos_pdfs()


Processando: Resultados da classificação.pdf...


Analisando matérias: 100%|██████████| 298/298 [00:00<00:00, 1052.86mat/s, cadastros=8543]



Processando: Resultados da classificação_.pdf...


Analisando matérias: 100%|██████████| 328/328 [00:00<00:00, 723.08mat/s, cadastros=20423]



Processando: Resultados da classificação__.pdf...


Analisando matérias: 100%|██████████| 215/215 [00:00<00:00, 679.21mat/s, cadastros=28169]



Processando: Resultados da classificação___.pdf...


Analisando matérias: 100%|██████████| 209/209 [00:00<00:00, 482.23mat/s, cadastros=38792]



✅ SUCESSO! [38792] Registros totais extraídos para o arquivo 'resultados_matriculados_geral.csv'.
